## impoorts

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys, time, warnings
sys.path.insert(0, "src")
warnings.filterwarnings("ignore")

from data import load_instance

RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
pd.set_option("display.width", 140)

# Experiment settings
CP_TIME_LIMIT = 120          # seconds, full instance
CP_TIME_LIMIT_SCALE = 60     # seconds, each scalability instance
GA_PARAMS = dict(pop_size=200, n_gen=400, cxpb=0.8, mutpb=0.3, indpb=0.05)
GA_SEEDS = range(10)         # independent GA runs on the full instance
SCALE_SIZES = [6, 10, 14, 18, 22]
SCALE_SEEDS = range(5)

## Data explore

In [8]:
raw = {f: pd.read_csv(f"data/{f}.csv") for f in
       ["schedule", "courses", "classrooms", "timeslots", "students", "instructors"]}
display(pd.DataFrame({"rows": {k: len(v) for k, v in raw.items()}}).T)

,schedule,courses,classrooms,timeslots,students,instructors
rows,9082,22,30,100,3000,50


In [9]:
inst = load_instance("data")
enrol = raw["schedule"][["student_id", "course_id"]].drop_duplicates()
per_student = enrol.groupby("student_id").size()
n_pairs = inst.n_exams * (inst.n_exams - 1) // 2
shared = [s for *_, s in inst.conflict_pairs()]

print(f"Exams (one per course)      : {inst.n_exams}")
print(f"Students with enrolments    : {enrol.student_id.nunique()}  (exams per student: "
      f"min {per_student.min()}, mean {per_student.mean():.2f}, max {per_student.max()})")
print(f"Students per exam           : min {inst.size.min()}, mean {inst.size.mean():.0f}, max {inst.size.max()}")
print(f"Rooms                       : {inst.n_rooms}, capacity {inst.room_cap.min()}-{inst.room_cap.max()}, "
      f"total seats {inst.room_cap.sum()}")
print(f"Conflicting exam pairs      : {len(shared)} of {n_pairs}  "
      f"(shared students per pair: {min(shared)}-{max(shared)})")

ts = raw["timeslots"]
bad = (ts.end_time <= ts.start_time).sum()
print(f"timeslots.csv rows with end <= start: {bad} of {len(ts)}  -> not usable as exam periods")

Exams (one per course)      : 22
Students with enrolments    : 3000  (exams per student: min 1, mean 3.03, max 5)
Students per exam           : min 376, mean 413, max 457
Rooms                       : 30, capacity 21-46, total seats 1004
Conflicting exam pairs      : 231 of 231  (shared students per pair: 30-79)
timeslots.csv rows with end <= start: 44 of 100  -> not usable as exam periods


In [ ]:
# cp = solve_cpsat(inst, time_limit=CP_TIME_LIMIT)